In [61]:
from dotenv import load_dotenv
load_dotenv()


True

In [62]:
from openai import OpenAI
openai_client = OpenAI()

In [63]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return response.output_text



# def llm(prompt):
#     response = openai_client.chat.completions.create(
#         model='gpt-4o-mini',
#         messages=[{"role": "user", "content": prompt}]
#     )
#     return response.choices[0].message.content

In [64]:
llm('hey whats up?')

'Hey! Not much—just here and ready to help. What’s up with you?'

In [65]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Yes—usually you can still join after the course has started, but it depends on the course’s enrollment policy and how far along it is.

If you want, send me:
- the course name
- the platform or school
- whether you’re asking as a student or participant

and I can help you figure out the best next step or draft a message to the instructor/admin.


In [66]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [67]:
# def llm(prompt):
#     response = openai_client.chat.completions.create(
#         model='gpt-4o-mini',
#         messages=[{"role": "user", "content": prompt}]
#     )
#     return response.choices[0].message.content

In [68]:
prompt = f'''
Your task is to answer questions from the course participants based on the 
provided context.
Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know".
Question:
{question}

Context:
{context}
'''

In [69]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [70]:
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

In [71]:
import requests
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [72]:
import requests
docs_url = 'https://datatalks.club/faq/json/llm-zoomcamp.json'
response = requests.get(docs_url)
course_llm_raw = response.json()
course_llm_raw

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer':

In [73]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [74]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [75]:
from minsearch import Index
index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [76]:
search_results = index.search(
    question,
    boost_dict={'question':2.0},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=3
    )
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [77]:
def search(question):
    return index.search(
        question,
        boost_dict={'question':2.0,'section':0.5},
        filter_dict={'course': 'llm-zoomcamp'},
        num_results=3
    )

In [78]:
def search(question,course='llm-zoomcamp'):
    boost_dict = {'question':2.0,'section':0.5}
    filter_dict = {'course': course}
    
    return index.search(
        question,
        boost_dict= boost_dict,
        filter_dict=filter_dict,
        num_results=3
    )

In [79]:
search_results = search(question)


We want to split it in two 

In [81]:
INSTRUCTIONS = ''' 
Your task is to answer questions from the course participants based on the 
provided context.
Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know".
'''

In [85]:
USER_PROMPT_TEMPLATE = f'''
Question:
{question}

Context:
{context}
'''

In [86]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [91]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [92]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.


In [93]:

response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
)

In [94]:
response.output_text

'Yes — you can join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [95]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_013f522e2f4ed3f2006a20d8d9dac081918dece70a298e79ac",
  "created_at": 1780537561.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_013f522e2f4ed3f2006a20d8da72d48191b2047db23b96daef",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1780537562.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "pre

In [96]:
response.usage

ResponseUsage(input_tokens=200, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=31, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=231)

In [97]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00028950000000000004

In [100]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
        model="gpt-5.4-mini",   
        input=message_history
)

In [101]:
response.output_text

'Yes, you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [102]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
            model=model,   
            input=message_history
    )
    return response.output_text

In [103]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    user_prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
answer = rag(question)
print(answer)